# 1 环境依赖

Use the environment created in Code2 `conda activate cellpose` 

# 2 数据目录结构

在每个slides的独立的文件夹下新建`segmentation/cellpose_output`文件夹来保存cellpose的输出

```
Experiment Folder
|
|---Slides1.ome.tif
|---Slides2.ome.tif
|---Slides3.ome.tif
...
|---Slides1
|   |---Tiles
|       |---FOV0.tif
|       |---FOV1.tif
|       ...
|   |---image_data
|       |---FOV0
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|       |---FOV1
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|   |---segementation
|       |---cellpose_input
|           |---FOV0.tif
|           |---FOV1.tif
|           ...
|       |---cellpose_output
|           |---FOV0.tif
|           |---FOV1.tif
|           ...
...
```

# 3 处理逻辑

## CellposeSAM
批量处理所有slides

# 4 原始代码

/home/jqyang/cellpose/notebook/CellposeSAM.ipynb

/home/jqyang/cellpose/notebook/cellposeSAM_batch.ipynb

In [1]:
import os
import glob
import numpy as np
import time
from cellpose import models, io, core, plot
import torch
from tqdm import tqdm

# ================= 配置区域 =================
INPUT_FOLDER = "../demo/B319D_rmbg/segmentation/cellpose_input"
OUTPUT_FOLDER = "../demo/B319D_rmbg/segmentation/cellpose_output"

CHANNELS = [1, 0] # [Cyto, Nuclei]
DIAMETER = 25   
FLOW_THRESHOLD = 0.8  
CELLPROB_THRESHOLD = 0.0
BATCH_SIZE = 32 
MODEL_TYPE = 'cpsam'
# ===========================================

# 1. 检查环境
use_gpu = core.use_gpu()
if use_gpu:
    print(f"✅ GPU activated: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU not found.")

# 2. 加载模型
print(f"Loading model '{MODEL_TYPE}'...")
model = models.CellposeModel(gpu=use_gpu, pretrained_model=MODEL_TYPE)

# 3. 读取文件
print(f"Reading images from {INPUT_FOLDER}...")
files = glob.glob(os.path.join(INPUT_FOLDER, "fov*.tiff"))
# 按数字排序
files.sort(key=lambda f: int(''.join(filter(str.isdigit, os.path.basename(f)))))

if not files:
    print("Error: No files found.")
else:
    imgs = []
    # 读取所有图片
    print("Loading images into memory...")
    for f in files:
        img = io.imread(f)
        imgs.append(img)
        
    print(f"Starting segmentation on {len(imgs)} images...")
    start_time = time.time()

    # 初始化结果列表
    masks = []
    flows = []

    # 4. 执行分割 (带进度条)
    for img in tqdm(imgs, desc="Segmenting FOVs", unit="img"):
        
        # 逐张处理
        m, f, s = model.eval(
            img, 
            batch_size=BATCH_SIZE, 
            diameter=DIAMETER,          
            flow_threshold=FLOW_THRESHOLD, 
            cellprob_threshold=CELLPROB_THRESHOLD,
            resample=False, 
            augment=False,
            do_3D=False
        )
        
        masks.append(m)
        flows.append(f)
    
    end_time = time.time()
    print(f"Segmentation complete. Total time: {end_time - start_time:.2f} s")

    # 5. 保存结果
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
        
    print(f"Saving masks to {OUTPUT_FOLDER}...")
    io.save_masks(
        imgs, 
        masks, 
        flows, 
        files, 
        channels=CHANNELS, 
        png=False, 
        tif=True, 
        savedir=OUTPUT_FOLDER,
        save_flows=False,
        save_outlines=False
    )
    print("Done! Variables 'files', 'imgs', 'masks', 'flows' are now ready for inspection.")

✅ GPU activated: NVIDIA RTX A4500
Loading model 'cpsam'...
Reading images from ../demo/B319D_rmbg/segmentation/cellpose_input...
Loading images into memory...
Starting segmentation on 64 images...


Segmenting FOVs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [01:53<00:00,  1.77s/img]


Segmentation complete. Total time: 113.40 s
Saving masks to ../demo/B319D_rmbg/segmentation/cellpose_output...


no masks found, will not save PNG or outlines
no masks found, will not save PNG or outlines
no masks found, will not save PNG or outlines


Done! Variables 'files', 'imgs', 'masks', 'flows' are now ready for inspection.
